# MATS Portfolio Backtest — Colab Runner

> **Just hit `Ctrl+F9` (Run All) and walk away.**
>
> For details, troubleshooting, and lessons learned, see [`INSTALLME.Colab.md`](https://github.com/aistudylearning/mats-code/blob/main/INSTALLME.Colab.md)

| Cell | What it does | Expected time |
|---|---|---|
| 1 — Setup | Install deps, mount Drive, clone repo, load data to SSD | ~3–5 min (zip exists) or ~5h (no zip, Drive fallback) |
| 2 — Backtest | Run 50-asset × 10-timeframe portfolio backtest | ~2–3 hours |
| 3 — Rescue | Manual report save if auto-copy failed | ~5 sec |

> ⚠️ **data.zip is created by `main.py fetch` on L1/L3 — not here.**
> If `data.zip` is missing on Drive, this cell falls back to reading directly
> from Drive (slower, ~5h backtest). Run `python3 main.py fetch` on L1 first.

---
**Anti-idle** — paste in DevTools (F12 → Console). Type `allow pasting` first if Chrome blocks it:
```javascript
setInterval(() => { document.querySelector('colab-connect-button').click(); }, 60000);
```

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║     Cell 1: MATS Colab Master Setup — run once per session      ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── Step 1: Install dependencies ────────────────────────────────────
# Do NOT pin pandas — pandas-ta manages its own version
!pip install -q polars pyarrow duckdb pandas-ta ccxt joblib

# ── Step 2: Mount Google Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Step 3: Clone or update the repo ────────────────────────────────
import os, time
if not os.path.exists('/content/mats-code'):
    !git clone https://github.com/aistudylearning/mats-code.git /content/mats-code
else:
    !cd /content/mats-code && git pull

# ── Step 4: Load data onto local SSD ────────────────────────────────
# data.zip is created by 'main.py fetch' on L1/L3 — Colab only consumes it.
DRIVE_DATA = "/content/drive/MyDrive/trading/raw/data/hot/data"
DRIVE_ZIP  = "/content/drive/MyDrive/trading/raw/data.zip"
LOCAL_ZIP  = "/content/data.zip"
LOCAL_DATA = "/content/data"

if os.path.exists(LOCAL_DATA):
    print("✅ Local data already present — skipping copy")

elif os.path.exists(DRIVE_ZIP):
    # Fast path: copy 1 zip file + unzip locally (~2–5 min total)
    print("✅ data.zip found on Drive — using fast ZIP bootstrap")
    !du -sh "{DRIVE_ZIP}"

    print("\n⏳ Copying ZIP to local SSD...")
    t0 = time.time()
    !cp "{DRIVE_ZIP}" "{LOCAL_ZIP}"
    print(f"✅ Copied in {(time.time()-t0)/60:.1f} min")

    print("⏳ Unzipping on local SSD...")
    t1 = time.time()
    !unzip -q "{LOCAL_ZIP}" -d "{LOCAL_DATA}"
    print(f"✅ Extracted in {(time.time()-t1)/60:.1f} min")

    os.remove(LOCAL_ZIP)  # Free Colab SSD space; Drive zip is kept
    print("🗑️  Local ZIP deleted (Drive copy kept for next session)")

else:
    # Slow fallback: read directly from Drive mount (~5h backtest)
    print("⚠️  WARNING: data.zip not found on Drive!")
    print("   → Run 'python3 main.py fetch' on L1/L3 to create it.")
    print("   → Falling back to Drive mount (backtest will be ~5h instead of ~2h)")
    LOCAL_DATA = DRIVE_DATA  # Point directly at Drive

# ── Step 5: Show data volume ─────────────────────────────────────────
total_bytes = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, files in os.walk(LOCAL_DATA) for f in files
)
file_count = sum(len(files) for _, _, files in os.walk(LOCAL_DATA))
print(f"\n📊 Data: {file_count:,} files | {total_bytes/1024/1024:.0f} MB")

# ── Step 6: Configure environment ───────────────────────────────────
os.environ["MATS_DATA_ROOT"] = LOCAL_DATA
%cd /content/mats-code

print(f"\n✅ MATS ready.")
print(f"   DATA_ROOT  : {os.environ['MATS_DATA_ROOT']}")
print(f"   Sample assets: {os.listdir(LOCAL_DATA)[:5]}")

---
## 🚀 Backtest

- **`-u`** forces unbuffered output → log appears in real time
- **`tee`** saves log to Drive AND shows it on screen simultaneously
- **HTML report** auto-saves to `My Drive → trading → reports` when done

> ⏱ Expected runtime on free Colab (2 vCPU): **~2–3 hours** (with local SSD) or **~5h** (Drive fallback)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║     Cell 2: Portfolio Backtest — 50 assets × 10 timeframes      ║
# ╚══════════════════════════════════════════════════════════════════╝

!python3 -u main.py portfolio \
    --signal 0.2 \
    --timeframe 1m 5m 15m 30m 1h 2h 4h 1D 1W 1M \
    --html \
    2>&1 | tee /content/drive/MyDrive/trading/reports/log_backtest_$(date +%Y%m%d_%H%M).txt

print("\n🏁 Backtest complete! Report auto-saved to Drive.")

---
## 🛟 Optional: Manual Report Rescue

Only run this if the HTML report did **not** appear in `My Drive → trading → reports`.

In [ ]:
# ── Only run if auto-copy failed ─────────────────────────────────────
import shutil, glob, os
saved = 0
for f in glob.glob('/content/mats-code/output/*.html'):
    dest = f"/content/drive/MyDrive/trading/reports/{os.path.basename(f)}"
    shutil.copy2(f, dest)
    print("✅ Saved:", dest)
    saved += 1
if saved == 0:
    print("⚠️  No HTML reports found in output/. Check if the backtest completed.")